# Predicción de procesos de compra sin adjudicación

## Entrega 2: descripción, exploración y preparación del dataset

El proyecto utiliza datos públicos de procesos de compra del sistema Buenos Aires Compras. Los archivos fueron descargados el 29 de octubre de 2026 y se encuentran organizados siguiendo la estructura de Cookiecutter Data Science.

El objetivo de esta etapa es conocer la estructura de los datos, identificar los tipos de variables, revisar valores faltantes y duplicados, y preparar una observación por cada proceso de compra antes de generar el modelo de aprendizaje automático.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Carga de los datos

Para el análisis se utilizan cuatro archivos relacionados. `main.csv` contiene la información principal de los procesos de compra, mientras que los demás archivos permiten conocer los renglones solicitados y cuáles fueron adjudicados.

In [4]:
procesos = pd.read_csv("../data/raw/main.csv")
adjudicaciones = pd.read_csv("../data/raw/awards.csv")
items_solicitados = pd.read_csv("../data/raw/tender_items.csv")
items_adjudicados = pd.read_csv("../data/raw/awards_items.csv")

In [5]:
procesos.head()

,_link,id,tag,date,ocid,language,initiationType,tender_id,tender_additionalProcurementCategories,tender_title,...,tender_enquiryPeriod_startDate,tender_enquiryPeriod_durationInDays,tender_procuringEntity_id,tender_procuringEntity_name,tender_techniques_frameworkAgreement_method,tender_techniques_frameworkAgreement_value_amount,tender_techniques_frameworkAgreement_value_currency,tender_techniques_frameworkAgreement_period_endDate,tender_techniques_frameworkAgreement_period_startDate,tender_techniques_frameworkAgreement_period_durationInDays
0,id-0.0,ocds-bulbcf-10002-0591-LPU22-3-0-2022-05-24T10...,compiled,2022-05-24T10:15:00-03:00,ocds-bulbcf-10002-0591-LPU22-3-0,es,tender,10002-0591-LPU22,Concesiones,Licitación Pública Bibliotecas de la Ciudad,...,2022-05-24T10:16:00-03:00,36,CABA-UE-10002,Direccion General Concesiones y Permisos,NaN,NaN,NaN,NaN,NaN,NaN
1,id-0.1,ocds-bulbcf-101-0316-CDI22-1-1-2022-04-05T12:0...,compiled,2022-04-05T12:00:00-03:00,ocds-bulbcf-101-0316-CDI22-1-1,es,tender,101-0316-CDI22,Servicios,Servicio para la realización de talleres a fin...,...,2022-04-05T12:30:00-03:00,1,CABA-UE-101,SECRETARIA LEGAL Y TECNICA,NaN,NaN,NaN,NaN,NaN,NaN
2,id-0.2,ocds-bulbcf-101-1835-LPU21-4-5-2022-02-07T12:0...,compiled,2022-02-07T12:00:00-03:00,ocds-bulbcf-101-1835-LPU21-4-5,es,tender,101-1835-LPU21,Equipamiento para Oficina,Adquisición de papel,...,2022-02-07T12:30:00-03:00,3,CABA-UE-101,SECRETARIA LEGAL Y TECNICA,NaN,NaN,NaN,NaN,NaN,NaN
3,id-0.3,ocds-bulbcf-112-0272-CME22-3-0-2022-03-31T14:0...,compiled,2022-03-31T14:00:00-03:00,ocds-bulbcf-112-0272-CME22-3-0,es,tender,112-0272-CME22,Seguridad,Provisión de servicios relacionados con el Si...,...,2022-03-31T15:00:00-03:00,7,CABA-UE-112,"CONSEJO DE LOS DERECHOS DE LAS NIÑAS, NIÑOS Y ...",NaN,NaN,NaN,NaN,NaN,NaN
4,id-0.4,ocds-bulbcf-112-0960-CME22-5-1-2022-04-26T14:0...,compiled,2022-04-26T14:00:00-03:00,ocds-bulbcf-112-0960-CME22-5-1,es,tender,112-0960-CME22,Construcción,Adquisicón de herramientas varias,...,2022-04-26T14:30:00-03:00,2,CABA-UE-112,"CONSEJO DE LOS DERECHOS DE LAS NIÑAS, NIÑOS Y ...",NaN,NaN,NaN,NaN,NaN,NaN


## 2. Dimensiones de los archivos

A continuación se observa la cantidad de filas y columnas de cada archivo utilizado. Las filas representan registros y las columnas representan las características disponibles.

In [6]:
print("main.csv:", procesos.shape)
print("awards.csv:", adjudicaciones.shape)
print("tender_items.csv:", items_solicitados.shape)
print("awards_items.csv:", items_adjudicados.shape)

main.csv: (23298, 33)
awards.csv: (13835, 12)
tender_items.csv: (23298, 11)
awards_items.csv: (13835, 12)


In [7]:
procesos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23298 entries, 0 to 23297
Data columns (total 33 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   _link                                                       23298 non-null  object 
 1   id                                                          23298 non-null  object 
 2   tag                                                         23298 non-null  object 
 3   date                                                        23298 non-null  object 
 4   ocid                                                        23298 non-null  object 
 5   language                                                    23298 non-null  object 
 6   initiationType                                              23298 non-null  object 
 7   tender_id                                                   23298 non-null  object 
 

### Interpretación inicial

El archivo principal contiene 23.298 registros y 33 columnas. Se identifican variables de texto, numéricas y booleanas. Las fechas se encuentran inicialmente almacenadas como texto, por lo que posteriormente deberán convertirse al tipo fecha.

La cantidad de registros todavía no representa necesariamente la cantidad de procesos de compra, ya que un mismo proceso puede contener varios renglones.

In [8]:
valores_faltantes = procesos.isnull().sum()
valores_faltantes[valores_faltantes > 0]

tender_status                                                    36
tender_procuringEntity_name                                      50
tender_techniques_frameworkAgreement_method                   20941
tender_techniques_frameworkAgreement_value_amount             22966
tender_techniques_frameworkAgreement_value_currency           22966
tender_techniques_frameworkAgreement_period_endDate           22966
tender_techniques_frameworkAgreement_period_startDate         23273
tender_techniques_frameworkAgreement_period_durationInDays    23273
dtype: int64

In [9]:
print("Filas duplicadas:", procesos.duplicated().sum())
print("Procesos de compra únicos:", procesos["tender_id"].nunique())

Filas duplicadas: 0
Procesos de compra únicos: 2950


## 3. Análisis de los estados de los procesos

Se analizan los estados disponibles para diferenciar los procesos finalizados de aquellos que todavía se encuentran abiertos o en evaluación.

In [10]:
procesos["tender_status"].value_counts(dropna=False)

tender_status
complete        16068
active           6057
cancelled         834
unsuccessful      303
NaN                36
Name: count, dtype: int64

In [12]:
estados_por_proceso = procesos[
    ["tender_id", "tender_status"]
].drop_duplicates(subset="tender_id")

estados_por_proceso["tender_status"].value_counts(dropna=False)

tender_status
complete        1896
active           665
cancelled        266
unsuccessful     114
NaN                9
Name: count, dtype: int64

### Interpretación de los estados

El dataset contiene 2.950 procesos de compra diferentes. De ellos, 1.896 figuran como finalizados, 665 continúan activos, 266 fueron cancelados, 114 finalizaron sin éxito y 9 no tienen estado informado.

Los procesos activos y aquellos sin estado no se utilizarán para construir la variable objetivo, porque todavía no se conoce con certeza su resultado.

In [14]:
adjudicaciones.columns

Index(['_link', '_link_main', 'id', 'date', 'status', 'title', 'description',
       'value_amount', 'value_currency', 'contractPeriod_endDate',
       'contractPeriod_startDate', 'contractPeriod_durationInDays'],
      dtype='object')

In [15]:
print("awards.csv:")
display(adjudicaciones[["_link_main", "id"]].head())

print("main.csv:")
display(procesos[["_link", "tender_id"]].head())

awards.csv:


,_link_main,id
0,id-0.1,101-0316-CDI22
1,id-0.2,101-1835-LPU21
2,id-0.3,112-0272-CME22
3,id-0.4,112-0960-CME22
4,id-0.6,2180-0165-LPU22


main.csv:


,_link,tender_id
0,id-0.0,10002-0591-LPU22
1,id-0.1,101-0316-CDI22
2,id-0.2,101-1835-LPU21
3,id-0.3,112-0272-CME22
4,id-0.4,112-0960-CME22


In [16]:
adjudicaciones_con_id = adjudicaciones[
    ["_link_main"]
].merge(
    procesos[["_link", "tender_id"]].drop_duplicates(subset="_link"),
    left_on="_link_main",
    right_on="_link",
    how="left"
)

ids_adjudicados = adjudicaciones_con_id[
    ["tender_id"]
].dropna().drop_duplicates()

ids_adjudicados["adjudicada"] = 1

datos_procesos = estados_por_proceso.merge(
    ids_adjudicados,
    on="tender_id",
    how="left"
)

datos_procesos["adjudicada"] = datos_procesos["adjudicada"].fillna(0)

datos_procesos.head()

,tender_id,tender_status,adjudicada
0,10002-0591-LPU22,active,0.0
1,101-0316-CDI22,complete,1.0
2,101-1835-LPU21,complete,1.0
3,112-0272-CME22,complete,1.0
4,112-0960-CME22,complete,1.0


In [17]:
datos_finales = datos_procesos[
    (datos_procesos["tender_status"] == "complete") |
    (datos_procesos["tender_status"] == "cancelled") |
    (datos_procesos["tender_status"] == "unsuccessful")
].copy()

datos_finales["adjudicada"] = datos_finales["adjudicada"].astype(int)

datos_finales["sin_adjudicacion"] = (
    datos_finales["adjudicada"] == 0
).astype(int)

datos_finales["sin_adjudicacion"].value_counts()

sin_adjudicacion
0    1757
1     519
Name: count, dtype: int64

## Conclusión de la preparación inicial

Se construyó una tabla con una observación por proceso de compra. Se excluyeron los procesos activos y aquellos sin estado, porque todavía no se conoce su resultado.

La variable objetivo `sin_adjudicacion` se definió de la siguiente manera:

- 1: el proceso finalizó sin ninguna adjudicación.
- 0: el proceso tuvo al menos una adjudicación.

Los procesos parcialmente adjudicados se consideran adjudicados, debido a que al menos uno de sus renglones obtuvo una adjudicación.Se profundizará la selección y transformación de las variables predictoras antes de generar los modelos.